In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import FastText, Word2Vec

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
MODEL_DIR = ROOT / "models"
models = {
    "CBOW": Word2Vec.load(str(MODEL_DIR / "cbow.model")),
    "Skip-gram": Word2Vec.load(str(MODEL_DIR / "skipgram.model")),
    "fastText": FastText.load(str(MODEL_DIR / "fasttext.model")),
}

In [2]:
queries = ["happy", "dog", "mom", "sad", "king"]
for q in queries:
    print(q)
    for name, m in models.items():
        print(f"  {name:9s}", [w for w, s in m.wv.most_similar(q, topn=5)])

happy
  CBOW      ['relieved', 'excited', 'pleased', 'lucky', 'glad']
  Skip-gram ['grateful', 'relieved', 'relieve', 'pleased', 'glad']
  fastText  ['relieved', 'grateful', 'pleased', 'cheerful', 'overjoyed']
dog
  CBOW      ['cat', 'spot', 'max', 'bone', 'puppy']
  Skip-gram ['cat', 'bark', 'spot', 'wag', 'barking']
  fastText  ['bark', 'cat', 'barking', 'wag', 'spot']
mom
  CBOW      ['mommy', 'dad', 'parent', 'mother', 'mum']
  Skip-gram ['dad', 'mommy', 'mum', 'daddy', 'parent']
  fastText  ['dad', 'mommy', 'mother', 'daddy', 'mum']
sad
  CBOW      ['upset', 'unhappy', 'guilty', 'anxious', 'frightened']
  Skip-gram ['upset', 'troubled', 'guilty', 'ashamed', 'helpless']
  fastText  ['upset', 'disappointed', 'unhappy', 'confused', 'guilty']
king
  CBOW      ['queen', 'prince', 'kingdom', 'wizard', 'person']
  Skip-gram ['queen', 'prince', 'kingdom', 'princess', 'knight']
  fastText  ['kingdom', 'queen', 'prince', 'knight', 'princess']


In [3]:
pairs = [("happy", "glad"), ("mom", "dad"), ("cat", "dog"), ("happy", "sad"), ("dog", "spoon"), ("king", "carrot")]
rows = []
for a, b in pairs:
    rows.append({"pair": f"{a} / {b}", **{name: round(float(m.wv.similarity(a, b)), 2) for name, m in models.items()}})
print(pd.DataFrame(rows).to_string(index=False))

         pair  CBOW  Skip-gram  fastText
 happy / glad  0.50       0.54      0.57
    mom / dad  0.80       0.81      0.79
    cat / dog  0.78       0.73      0.75
  happy / sad  0.44       0.46      0.43
  dog / spoon  0.17       0.15      0.22
king / carrot  0.02       0.21      0.21


In [4]:
tests = [(["king", "woman"], ["man"]), (["brother", "girl"], ["boy"])]
for pos, neg in tests:
    print(f"{pos[0]} - {neg[0]} + {pos[1]}")
    for name, m in models.items():
        top = m.wv.most_similar(positive=pos, negative=neg, topn=3)
        print(f"  {name:9s}", [(w, round(s, 2)) for w, s in top])

king - man + woman
  CBOW      [('prince', 0.7), ('queen', 0.69), ('wizard', 0.59)]
  Skip-gram [('princess', 0.63), ('prince', 0.61), ('queen', 0.58)]
  fastText  [('prince', 0.64), ('princess', 0.62), ('queen', 0.62)]
brother - boy + girl
  CBOW      [('aunt', 0.54), ('doll', 0.53), ('emma', 0.53)]
  Skip-gram [('sister', 0.55), ('her', 0.52), ('doll', 0.5)]
  fastText  [('sister', 0.54), ('her', 0.54), ('doll', 0.51)]


In [5]:
from src.vectorize import METHODS, vectorize

raw = """Features: Dialogue
Story: Tom had a red ball. "Can I play?" asked his sister. Tom smiled and said, "Yes, let's play together!"
<|endoftext|>"""
rows = []
for m in METHODS:
    v = vectorize(raw, method=m)
    rows.append({"method": m, "length": v.shape[0], "non-zero %": round(100 * np.count_nonzero(v) / v.shape[0], 1)})
print(pd.DataFrame(rows).to_string(index=False))

  method  length  non-zero %
     bow    6298         0.3
   tfidf    6298         0.3
    cbow     100       100.0
skipgram     100       100.0
fasttext     100       100.0


In [6]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

a = "The little dog ran in the park with a ball."
b = "A puppy played fetch with a ball in the garden."
c = "The king lived in a big castle."
for m in ["tfidf", "skipgram"]:
    va, vb, vc = (vectorize(t, m) for t in (a, b, c))
    print(f"{m:9s} dog~puppy: {cosine(va, vb):.2f}   dog~king: {cosine(va, vc):.2f}")

tfidf     dog~puppy: 0.29   dog~king: 0.07


skipgram  dog~puppy: 0.93   dog~king: 0.72
